# 🌱 Simulation of Wheat Vegetative Growth with CNW-Grass based on CN metabolism

[CNW-Grass](https://cnwgrass.readthedocs.io/en/latest/?badge=latest) is a Functional Structural Plant Model (FSPM) of wheat which fully integrates plant functioning and shoot morphogenesis with carbon and nitrogen metabolism and water flows at the organ level. Plants are described in 3D as a collection of tillers, each consisting in individual shoot organs (lamina, sheath, internode, peduncle, chaff), a single root compartment, the grains and a phloem.

The first example below illustrates how to use the CNW-Grass model to simulate the growth and functioning of a wheat plant during the vegetative stages. Only the metabolic regulations are taken into account. This example is based on the script [main.py](https://github.com/openalea/cnwgrass/blob/master/example/Vegetative_stages/main.py).

## 🛠️ Configuration and prerequisites
Let's import all the required packages

In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# Adel-Wheat dependencies (https://github.com/openalea/adel)
from openalea.adel.adel_dynamic import AdelDyn
from openalea.adel.echap_leaf import echap_leaves

from openalea.widgets.plantgl import PlantGL

## Model inputs
* Weather data

   They should contain the following variables :
    * *t*: the time index used in the simulation (hour)
    * *DOY*: the Day Of the Year
    * *hour*: the hour of the day [1-24]
    * *air_temperature*: temperature of the air (°C)
    * *soil_temperature*: temperature of the soil (°C)
    * *PARi*: incident Photosynthetically Active Radiation above the canopy (µmol m-2 s-1)
    * *humidity*: air humidity (relative)
    * *ambient_CO2*: CO2 concentration of the atmosphere (ppm)
    * *Wind* : wind speed 2m above the canopy (m s-1)


In [2]:
INPUTS_DIRPATH='inputs'

# Loading a meteo file and display head
METEO_FILENAME = "meteo_file.csv"
meteo_df = pd.read_csv(os.path.join(INPUTS_DIRPATH, METEO_FILENAME), delimiter=',',  index_col='t')
meteo_df.head()

,Date,DOY,hour,air_temperature,PARi,soil_temperature,humidity,ambient_CO2,Wind
t,,,,,,,,,
0,17/12/1998,351,12,5.2,516.222222,5.840,0.95,360,2.9
1,17/12/1998,351,13,5.6,549.888889,6.264,0.88,360,2.9
2,17/12/1998,351,14,4.8,392.777778,5.415,0.92,360,2.9
3,17/12/1998,351,15,3.6,157.111111,6.223,0.95,360,2.9
4,17/12/1998,351,16,3.5,22.444444,5.957,0.96,360,2.9


* Setting management:

   The management options should be set in a csv file. The file should have the following variables:
   * *plant_density*: the sowing density per genotype in plants per square meter. Provide it as a dict {genotype_id: sowing_density_value}
   * *N_fertilizations*: the fertilization regime in µmol of N). Provide it as a dict {hour of the simulation: fertilisation_value}

In [4]:
MANAGEMENT_FILENAME = 'management.csv'
print(pd.read_csv(os.path.join(INPUTS_DIRPATH, MANAGEMENT_FILENAME)))

               name                          value
0     plant_density                       {1: 250}
1  N_fertilizations  {1440: 357143, 2520: 1000000}


* Initial conditions

  The initial state of the different plant organs and the soil are loaded from different csv files. They mainly contain the initial dimensions, masses and metabolic and water contents. By default, the files with the different initial states are located in a subfolder named *inputs*.

In [5]:
# Name of the CSV files which describes the initial state of the system
AXES_INITIAL_STATE_FILENAME = 'axes_initial_state.csv'  # Axis scale
ORGANS_INITIAL_STATE_FILENAME = 'organs_initial_state.csv'  # Organ scale (contains data for the roots and phloem compartments)
HIDDENZONES_INITIAL_STATE_FILENAME = 'hiddenzones_initial_state.csv'  # Hidden zone scale
ELEMENTS_INITIAL_STATE_FILENAME = 'elements_initial_state.csv'  # Photosynthetic element scale
SOILS_INITIAL_STATE_FILENAME = 'soils_initial_state.csv'  # Soil


# Read the inputs from CSV files and create inputs dataframes

for inputs_filename in (AXES_INITIAL_STATE_FILENAME,
                        ORGANS_INITIAL_STATE_FILENAME,
                        HIDDENZONES_INITIAL_STATE_FILENAME,
                        ELEMENTS_INITIAL_STATE_FILENAME,
                        SOILS_INITIAL_STATE_FILENAME):
    inputs_dataframe = pd.read_csv(os.path.join(INPUTS_DIRPATH, inputs_filename))
    print(inputs_filename, '\n', inputs_dataframe.head())

axes_initial_state.csv 
    plant axis      status  nb_leaves     GA  SAM_height  SAM_temperature  \
0      1   MS  vegetative         10  False           0               20   
1      1   T1  vegetative          7  False           0               20   
2      1   T2  vegetative          6  False           0               20   
3      1   T3  vegetative          5  False           0               20   
4      1   T4  vegetative          4  False           0               20   

   delta_teq  delta_teq_roots  teq_since_primordium  sum_TT  cohort   mstruct  \
0       3600             3600                     0       0       1  0.054681   
1       3600             3600                     0       0       4       NaN   
2       3600             3600                     0       0       5       NaN   
3       3600             3600                     0       0       6       NaN   
4       3600             3600                     0       0       7       NaN   

   senesced_mstruct  minerals_p

Note the decomposition of the plant into axes, metamers, organs and elements. More information can be found in the [documentation](https://cnwgrass.readthedocs.io/en/latest/?badge=latest).

* Configuration of Adel-Wheat

  CNW-Grass includes an implicit coupling to the model [Adel-Wheat](https://adel.readthedocs.io/en/latest/?badge=latest). This models allows for the 3D representation of the wheat architecture and the generation of the MTG object used for the communication (read/write) between each submodule of CNW-Grass. The inputs should therefore contain a pickle of an Adel MTG (adel0000.pckl and scene0000.bgeom) and two tables to set the plant architecture in Adel (adel_pars.RData and phytoT.csv).

  In this example, we will use the leaf database for the wheat cultivar Soissons.

In [6]:
#- Illustration of the initial MTG used the vegetative stages example (starts at leaf 4 emergence).
# Instantiates AdelDyn object with Soissons database and set scene unit to meter
adel_wheat = AdelDyn(seed=1, scene_unit='m', leaves=echap_leaves(xy_model='Soissons_byleafclass'))
# Instantiates a MTG object from a pickle
g = adel_wheat.load(directory=INPUTS_DIRPATH)

# Visualisation of the input wheat architecture
scene = adel_wheat.scene(g)
mesh=PlantGL(scene)
mesh

Plot(antialias=3, axes=['x', 'y', 'z'], axes_helper=1.0, axes_helper_colors=[16711680, 65280, 255], background…

Exploring the MTG
* The phloem, roots and grains compartment are stored at the axis vertex of the MTG. The different metabolic, dimension and mass related-properties are stored as properties of those compartments.
* The hiddenzone compartments are stored at each metamer vertex of the MTG.
* Each metamer is  characterized by 3 MTG components : a blade, a sheath and an internode
* Each organ can be composed by 2 elements, which are used to separate the hidden and visible part of the organ

* Configuration of outputs and postprocessing.

  By default, the model will write raw outputs (one csv file per scale) in a folder *outputs* and postprocessing (one csv file per scale) in a folder *postprocessing*.

In [8]:
# -- OUTPUT CONFIGURATION --
OUTPUTS_DIRPATH = os.path.join('outputs', 'simple_run_vegetative_stages')

# -- POSTPROCESSING CONFIGURATION --
POSTPROCESSING_DIRPATH= os.path.join('postprocessing', 'simple_run_vegetative_stages')

# -- POSTPROCESSING CONFIGURATION --
GRAPH_DIRPATH= os.path.join('graphs', 'simple_run_vegetative_stages')

## Run of the simulation
Running the simulation from the *main.py* by calling *openalea.cnwgrass.integration.runner*.

In [9]:
from openalea.cnwgrass.integration.runner import run as runner

SIMULATION_LENGTH = 48  # length of the simulation (hours)
START_TIME = 0

# precision of floats used to write and format the output CSV files
OUTPUTS_PRECISION = 2

# Rates and time-dependant parameters in CNW-Grass are expressed in s or s-1 but simulations (and differential equations solve) are usually integrated over an hour
HOUR_TO_SECOND_CONVERSION_FACTOR = 3600

runner(simulation_length=SIMULATION_LENGTH, forced_start_time=0,
                run_simu=True, run_postprocessing=True, generate_graphs=True, run_from_outputs=False,
                tillers_replications={'T1': 0.5, 'T2': 0.5, 'T3': 0.5, 'T4': 0.5}, heterogeneous_canopy=True,
                METEO_FILENAME=METEO_FILENAME, MANAGEMENT_FILENAME=MANAGEMENT_FILENAME,
                INPUTS_DIRPATH=INPUTS_DIRPATH, OUTPUTS_DIRPATH=OUTPUTS_DIRPATH, POSTPROCESSING_DIRPATH=POSTPROCESSING_DIRPATH, GRAPHS_DIRPATH=GRAPH_DIRPATH)



NameError: name 'openalea' is not defined

## Generate outputs, postprocessing and graphs
Outputs relate to the different variable solved by the model, while the postprocessing include extra-calculations allowing for in-depth interpretation of the simulations. Outputs and postprocessing are written as csv files, one per scale (axis, organs, hiddenzones, elements and soil). Files are stored in /outputs/simple_run_vegetative_stages and /postprocessing/simple_run_vegetative_stages.
At the end of the simulation, main prags are automatically plotted and stored in the specified folder

## 📈 Key Results Visualization

In [ ]:
axes_postprocessing_df = pd.read_csv(os.path.join(POSTPROCESSING_DIRPATH, 'axes_postprocessing.csv'))
organs_postprocessing_df = pd.read_csv(os.path.join(POSTPROCESSING_DIRPATH, 'organs_postprocessing.csv'))
roots_postprocessing_df = organs_postprocessing_df[organs_postprocessing_df['organ'] == 'roots']
hz_postprocessing_df = pd.read_csv(os.path.join(POSTPROCESSING_DIRPATH, 'hiddenzones_postprocessing.csv'))
hz_MS_postprocessing_df = hz_postprocessing_df[hz_postprocessing_df['axis'] == 'MS']

# Biomass allocation
plt.figure(figsize=(10, 6))
plt.plot(axes_postprocessing_df['t'], axes_postprocessing_df['sum_dry_mass_roots'], label='Root biomass')
plt.plot(axes_postprocessing_df['t'], axes_postprocessing_df['sum_dry_mass_shoot'], label='Shoot biomass')
plt.xlabel('Time (hour)')
plt.ylabel('Dry mass (g)')
plt.title('Biomass allocation')
plt.legend()
plt.grid(True)
plt.show()

# C and N acquisition
fig, ax1 = plt.subplots(figsize=(10, 6))
ax1.plot(axes_postprocessing_df['t'], axes_postprocessing_df['Total_Photosynthesis'], label='Photosynthesis', color='yellow')
ax1.set_xlabel('Time (hour)')
ax1.set_ylabel('Photosynthesis (µmol C)')

ax2 = ax1.twinx()
ax2.plot(roots_postprocessing_df['t'], roots_postprocessing_df['Uptake_Nitrates'], label='N uptake', color='green')
ax2.set_ylabel('N Uptake (µmol N)')

plt.title('C and N acquisition')
fig.legend(loc='upper right')
plt.grid(True)
plt.show()

# Leaf length
plt.figure(figsize=(10, 6))
for metamer in hz_MS_postprocessing_df['metamer'].unique():
    leaf = hz_MS_postprocessing_df[hz_MS_postprocessing_df['metamer'] == metamer]
    plt.plot(leaf['t'], leaf['leaf_L'], label=metamer)

plt.xlabel('Time (hour)')
plt.ylabel('Leaf length (m)')
plt.legend(loc='center right', title='Leaf number')
plt.grid(True)
plt.show()
